# 🧬 RFdiffusion3 Tool Example

This notebook demonstrates how to use **RFdiffusion3 (RFdiffusion3)** for de novo protein structure design.

## 📖 What is RFdiffusion3?

[RFdiffusion3](https://github.com/RosettaCommons/foundry) is an all-atom diffusion model for generating new protein structures and matching sequences under user-defined constraints.

### Key Features:
- **Structure generation** -- Generate novel 3D protein structures de novo
- **Sequence co-design** -- Output sequence and structure together
- **Flexible conditioning** -- Support motif scaffolding, hotspot-guided design, and binder workflows
- **Configurable sampling** -- Control diffusion steps, batch size, and diversity/designability balance

## 📥 Imports

## 📦 Shared Data Types

### `Structure`
A predicted 3D structure with coordinates, confidence metrics, and export methods.

### `RFdiffusion3DesignSpec`
Specification for a single design task.

| Field | Type | Description |
|-------|------|-------------|
| `input_structure` | `Optional[str]` | Path to PDB/CIF file or PDB string |
| `contig` | `Optional[str]` | Design topology (e.g., `"50-80"`, `"A1-100,50,A150-200"`) |
| `length` | `Optional[str]` | Length constraint (integer or `"min-max"` range) |
| `ligand` | `Optional[str]` | Ligand selection (3-letter codes) |
| `unindex` | `Optional[Union[str, Dict]]` | Unindexed motif components |
| `select_fixed_atoms` | `Optional[Union[bool, str, Dict]]` | Atoms to fix in place |
| `select_unfixed_sequence` | `Optional[Union[bool, str, Dict]]` | Residues whose sequence can change |
| `select_hotspots` | `Optional[Union[str, Dict]]` | Interaction hotspot residues |
| `partial_t` | `Optional[float]` | Noise level in Angstroms (5.0--15.0 recommended) |

In [1]:
from proto_tools import (
    RFdiffusion3Config,
    RFdiffusion3DesignSpec,
    RFdiffusion3Input,
    run_rfdiffusion3,
)


## 🧪 1. Configure and Run Design

Generate a 40-residue protein from scratch using unconditional design.

Running a design requires two objects passed to `run_rfdiffusion3(inputs, config)`:

1. **`RFdiffusion3Input`** -- *what* to design. Contains a list of `RFdiffusion3DesignSpec` objects, where each spec defines one independent design task (target length, contig topology, input structure, etc.). Multiple specs in a single input are processed together in one run.
2. **`RFdiffusion3Config`** -- *how* to sample. Controls diffusion parameters like batch size, timesteps, and diversity.

### Design specification (`RFdiffusion3DesignSpec`)

Each spec defines one design task -- its target topology, constraints, and conditioning:

- **`length`** -- Target length, e.g. `"40"` or `"80-120"` (for unconditional design)
- **`contig`** -- Contig topology string for motif scaffolding or multi-chain designs
- **`input_structure`** -- Input PDB/CIF path for conditioned design (binders, scaffolding)
- **`ligand`** -- Ligand constraints by residue name, e.g. `"HEM"`
- **`select_hotspots`** -- Interface hotspots for binder design
- **`partial_t`** -- Noise level (Angstroms) for partial diffusion refinement

### API Reference

**`RFdiffusion3Input`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `design_specs` | `List[RFdiffusion3DesignSpec]` | `[]` | List of design specifications, each defining an independent design task |
| `raw_json` | `Optional[str]` | `None` | Raw JSON string for advanced RFdiffusion3 configuration (bypasses `design_specs`) |

**`RFdiffusion3Config`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `n_batches` | `int` | `1` | Number of batches to generate per input key |
| `diffusion_batch_size` | `int` | `8` | Number of diffusion samples (designs) per batch |
| `num_timesteps` | `int` | `200` | Diffusion timesteps for sampling; higher = slower but often better |
| `step_scale` | `float` | `1.5` | Diversity vs designability tradeoff; higher = more conservative |
| `low_memory_mode` | `bool` | `False` | Memory-efficient tokenization mode for limited GPU RAM |

Total designs generated = `n_batches` x `diffusion_batch_size` x number of specs.

**`RFdiffusion3Output`**

| Field | Type | Description |
|-------|------|-------------|
| `output_structures` | `List[RFdiffusion3Structure]` | List of designed structures |

Supports list-like access: indexing (`output[0]`), iteration (`for s in output`), and `len(output)`.

**`RFdiffusion3Structure`**

| Field | Type | Description |
|-------|------|-------------|
| `structure` | `Structure` | The designed 3D structure with atomic coordinates |
| `sequence` | `str` | The designed amino acid sequence (chains separated by `/` for multi-chain) |
| `spec_key` | `str` | Identifier of the input spec that produced this design (e.g., `"spec-0"`) |
| `design_index` | `int` | Index of this design within its batch |
| `metadata` | `Dict[str, Any]` | Additional metadata (sampled contig, chain info, metrics, etc.) |

> **Tip:** For advanced Foundry options not exposed by `RFdiffusion3DesignSpec`, you can bypass the structured API entirely with `RFdiffusion3Input(raw_json=...)`, which passes a raw JSON string directly to RFdiffusion3.

In [2]:
inputs = RFdiffusion3Input(
    design_specs=[
        RFdiffusion3DesignSpec(length="100")
    ]
)

# Minimal config for a quick test run
config = RFdiffusion3Config(
    n_batches=1,
    diffusion_batch_size=1,
    num_timesteps=50,
    step_scale=1.5,
)

output = run_rfdiffusion3(inputs, config)
print(f"Generated {len(output)} structures")


Generated 1 structures


## 🔍 2. Inspect Designs

Inspect the generated sequence and metadata for the first design.


In [3]:
first = output[0]
print(f"Spec key: {first.spec_key}")
print(f"Design index: {first.design_index}")
print(f"Sequence length: {len(first.sequence)}")
print(f"Sequence: {first.sequence}")
print(f"Metadata keys: {list(first.metadata.keys())}")

Spec key: spec-0
Design index: 0
Sequence length: 100
Sequence: AICACTNLASIPELQALGNLSALVLSASNTSLYVTVSPGSAAFTPLTVSLLNANGALVASGTTSAAGAAQLANLSLNTSYTLQIKQASSSVVSSPLAVTT
Metadata keys: ['diffused_index_map', 'metrics', 'specification', 'ckpt_path', 'seed']


## 🎨 3. Visualize Designs

Visualize the generated protein structures in 3D before exporting.

In [4]:
output[0].structure.visualize()

## 💾 4. Export Results

Save generated structures for downstream analysis.

### Supported formats:
- **PDB** -- Standard structure format for visualization and downstream tools
- **CIF** -- mmCIF structure format

In [5]:
output.export("example_output/rfdiffusion3_designs_pdb", file_format="pdb")
print("PDB designs exported to example_output/rfdiffusion3_designs_pdb/")

output.export("example_output/rfdiffusion3_designs_cif", file_format="cif")
print("CIF designs exported to example_output/rfdiffusion3_designs_cif/")

PDB designs exported to example_output/rfdiffusion3_designs_pdb/
CIF designs exported to example_output/rfdiffusion3_designs_cif/
